## Лабораторна робота Зашкольної Дарії, компмат2, Варіант 2

In [150]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [151]:
data = pd.read_csv("data/adult.data", header=None)
data.columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income"
]

data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [152]:
data.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

In [153]:
for i in data.columns :
    print (i, '\t', len(set(data[i])))

age 	 73
workclass 	 9
fnlwgt 	 21648
education 	 16
education-num 	 16
marital-status 	 7
occupation 	 15
relationship 	 6
race 	 5
sex 	 2
capital-gain 	 119
capital-loss 	 92
hours-per-week 	 94
native-country 	 42
income 	 2


Аналізуєм дані: дивимся скільки унікальних значень для кожного стовпчика, які з них категоріальні, а які числові. Оскільки пропусків в нас немає, а невизначених даних небагато ("?"), можемо просто їх замінити на None. В заміні на середнє немає сенсу, оскільки всі невизначеності стоять в категоріальних змінних. 

In [154]:
data = data.replace(" ?", np.nan)

data = pd.get_dummies(data)
data = data.dropna()

data.head()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,...,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia,income_ <=50K,income_ >50K
0,39,77516,13,2174,0,40,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
1,50,83311,13,0,0,13,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
2,38,215646,9,0,0,40,False,False,False,True,...,False,False,False,False,False,True,False,False,True,False
3,53,234721,7,0,0,40,False,False,False,True,...,False,False,False,False,False,True,False,False,True,False
4,28,338409,13,0,0,40,False,False,False,True,...,False,False,False,False,False,False,False,False,True,False


Кодуєм текстові дані. Ознака education є факторною, тому для коректної роботи моделей було застосовано One-Hot. Це дозволяє уникнути штучного порядку між категоріями та покращує якість моделей, особливо для логістичної регресії, SVM та k-NN. Після цього отримуєм датафрейм з 107 стовпців, бо всі дані в категоріях розклалися на класи. Далі розбиваємо дані на тестові і навчальні.

In [155]:
X = data.drop(["income_ <=50K", "income_ >50K", "fnlwgt"], axis=1)
y = data["income_ <=50K"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Важливий момент: видаляємо стовпчик fnlwgt. Можемо помітити, що в нього надто багато різних даних, це якийсь ідентифікатор чи щось типу, він не несе прямої інформації для класифікації, ба більше, може погано впливати на нашу модель (перенавчання). Тому видаляємо його. 

In [156]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Масштабуєм всі ознаки, щоб зробити їх порівнянними

In [157]:
nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))

Naive Bayes: 0.4082604022723783


Низький показник. Наївний Баєс припускає, що всі ознаки незалежні та мають нормальний розподіл. А в нас після get_dummies більшість ознак бінарні, і до того ж кореляція між деякими ознаками порушує припущення незалежності. Модель "вчиться" неправильним закономірностям. Висновок: для даних з багатьма бінарними ознаками наївний Баєс не підходить.

In [158]:
for k in [3, 5, 7, 10]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    print(f"KNN (k={k}):", accuracy_score(y_test, y_pred))

KNN (k=3): 0.8200522032857362
KNN (k=5): 0.8281897742975587
KNN (k=7): 0.8323353293413174
KNN (k=10): 0.8334101028711807


Хороший результат. Бачимо, що зі зростанням k, результат поліпшується. Це може означати, що модель чутлива до викидів, і зі збільшенням сусідів точність збільшується, бо шум згладжується.

In [159]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))

Logistic Regression: 0.857362198679564


Найкращий результат з усіх попередніх. Що логічно, бо логістична регресія працює без припущень про розподіли і нормально справляється з такими даними.

In [160]:
param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto']
}

svm = SVC()

grid = GridSearchCV(svm, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)

Best params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}


Підбираємо гіперпараметри через GridSearchCV з трьома ядрами (лінійне, RBF, поліноміальне), різними значеннями регуляризації C та параметрами gamma. Найкращі значення вивели і на їхній основі далі навчаємо SVM

In [161]:
svm = SVC(C=0.1, kernel='linear')
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print(f"SVM ({kernel}):", accuracy_score(y_test, y_pred))

SVM (rbf): 0.8565945033010901


In [162]:
# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
print("Decision Tree:", accuracy_score(y_test, dt.predict(X_test)))

# Впадковий ліс
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
print("Random Forest:", accuracy_score(y_test, rf.predict(X_test)))

# Градієнтний бустінг
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
print("Gradient Boosting:", accuracy_score(y_test, gb.predict(X_test)))

Decision Tree: 0.8182097343773991
Random Forest: 0.852141870105942
Gradient Boosting: 0.8713342545677875


Decision Tree (0.818) - найслабший серед трьох. Дерево рішень без обмеження глибини має схильність до сильного перенавчання, але тут воно показує помірну точність. Це означає, що дані не надто складні, але й не тривіальні. Random Forest (0.852) - значно краще за попереднє дерево. Усереднення 100 дерев зменшує дисперсію та покращує узагальнення. Gradient Boosting (0.871) найкращий результат серед усіх методів. Бустинг послідовно виправляє помилки попередніх дерев, тому він часто дає вищу точність, ніж Random Forest.

In [163]:
def apply_model(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    print(f"\n=== {name} ===")
    
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_test_pred))
    
    metrics = {
        "accuracy": accuracy_score(y_test, y_test_pred),
        "precision": precision_score(y_test, y_test_pred),
        "recall": recall_score(y_test, y_test_pred),
        "f1": f1_score(y_test, y_test_pred)
    }
    
    df_metrics = pd.DataFrame(metrics, index=[name])
    print(df_metrics)
    
    return df_metrics

In [165]:
results = []

results.append(apply_model(GaussianNB(), X_train, y_train, X_test, y_test, "Naive Bayes"))

results.append(apply_model(KNeighborsClassifier(n_neighbors=10), X_train, y_train, X_test, y_test, "KNN"))

results.append(apply_model(LogisticRegression(max_iter=1000), X_train, y_train, X_test, y_test, "Logistic"))

results.append(apply_model(SVC(kernel='linear'), X_train, y_train, X_test, y_test, "SVM"))

results.append(apply_model(DecisionTreeClassifier(), X_train, y_train, X_test, y_test, "Decision Tree"))

results.append(apply_model(RandomForestClassifier(), X_train, y_train, X_test, y_test, "Random Forest"))

results.append(apply_model(GradientBoostingClassifier(), X_train, y_train, X_test, y_test, "Boosting"))


=== Naive Bayes ===
Confusion matrix:
[[1539   32]
 [3822 1120]]
             accuracy  precision    recall        f1
Naive Bayes   0.40826   0.972222  0.226629  0.367575

=== KNN ===
Confusion matrix:
[[ 976  595]
 [ 490 4452]]
     accuracy  precision   recall        f1
KNN   0.83341   0.882108  0.90085  0.891381

=== Logistic ===
Confusion matrix:
[[ 962  609]
 [ 320 4622]]
          accuracy  precision    recall       f1
Logistic  0.857362   0.883579  0.935249  0.90868

=== SVM ===
Confusion matrix:
[[ 930  641]
 [ 299 4643]]
     accuracy  precision    recall        f1
SVM  0.855673    0.87869  0.939498  0.908077

=== Decision Tree ===
Confusion matrix:
[[1037  534]
 [ 653 4289]]
               accuracy  precision    recall        f1
Decision Tree  0.817749   0.889281  0.867867  0.878443

=== Random Forest ===
Confusion matrix:
[[1000  571]
 [ 403 4539]]
               accuracy  precision    recall        f1
Random Forest  0.850453   0.888258  0.918454  0.903104

=== Boosting ===

In [166]:
final_results = pd.concat(results)
final_results

,accuracy,precision,recall,f1
Naive Bayes,0.408260,0.972222,0.226629,0.367575
KNN,0.833410,0.882108,0.900850,0.891381
Logistic,0.857362,0.883579,0.935249,0.908680
SVM,0.855673,0.878690,0.939498,0.908077
Decision Tree,0.817749,0.889281,0.867867,0.878443
Random Forest,0.850453,0.888258,0.918454,0.903104
Boosting,0.871334,0.887755,0.950627,0.918116


Робим підсумкові таблички, щоб порівняти всі використані методи. В останній таблиці наведено чотири метрики якості для семи моделей на тестових даних. Точність (accuracy) показує частку правильно класифікованих об'єктів, precision - точність передбачення позитивного класу (дохід більше 50K), recall - чутливість (частку реальних позитивних об'єктів, які модель знайшла), F1 - гармонійне середнє між precision та recall. Найкращі результати демонструє Gradient Boosting з accuracy 0.871, recall 0.951 та F1 0.918. Це означає, що бустинг правильно передбачає високий дохід у 95% випадків.

Найгірший результат у наївного Баєса (0.408). Низький recall (0.227) вказує, що модель знаходить лише 23% реальних позитивних прикладів, хоча precision дуже високий (0.972)

У цій роботі категоріальні змінні було перетворено на фіктивні (one-hot encoding). Цей підхід добре працює для логістичної регресії, SVM та дерев, оскільки всі вони підтримують числові вхідні дані. Альтернативним методом, який міг би дати кращі результати для факторів із великою кількістю категорій (наприклад, native-country з 42 значеннями), є розбиття вибірки на підвибірки за категоріями та побудова окремих моделей. Однак через обмеження обсягу вибірки такий підхід може призвести до перенавчання, тому було обрано класичне кодування.

In [169]:
# Decision Tree з різною глибиною
print("=== Decision Tree (різні глибини) ===")
for depth in [None, 5, 10, 15]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Decision Tree (depth={depth}): Accuracy={acc:.4f}")

# з різною кількістю дерев
print("\n=== Random Forest (різна кількість дерев) ===")
for n in [50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Random Forest (n={n}): Accuracy={acc:.4f}")

# з різною швидкістю навчання
print("\n=== Gradient Boosting (різна швидкість навчання) ===")
for lr_rate in [0.05, 0.1, 0.2]:
    gb = GradientBoostingClassifier(learning_rate=lr_rate, n_estimators=100, random_state=42)
    gb.fit(X_train, y_train)
    y_pred = gb.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Gradient Boosting (lr={lr_rate}): Accuracy={acc:.4f}")

=== Decision Tree (різні глибини) ===
Decision Tree (depth=None): Accuracy=0.8201
Decision Tree (depth=5): Accuracy=0.8505
Decision Tree (depth=10): Accuracy=0.8589
Decision Tree (depth=15): Accuracy=0.8485

=== Random Forest (різна кількість дерев) ===
Random Forest (n=50): Accuracy=0.8520
Random Forest (n=100): Accuracy=0.8505
Random Forest (n=200): Accuracy=0.8534

=== Gradient Boosting (різна швидкість навчання) ===
Gradient Boosting (lr=0.05): Accuracy=0.8632
Gradient Boosting (lr=0.1): Accuracy=0.8713
Gradient Boosting (lr=0.2): Accuracy=0.8738


Decision Tree: найкраща глибина - 10 (точність 0.8589). Random Forest: всі значення кількості дерев дають стабільну точність 0.850-0.853. Різниця між 50, 100 та 200 деревами мінімальна, що свідчить про стійкість методу. Оптимальне значення - 200 дерев (0.8534). Gradient Boosting: точність зростає зі збільшенням швидкості навчання (learning rate). Найкращий результат - 0.8738 при lr=0.2.